
databricks free https://dbc-0b69ef93-5247.cloud.databricks.com/editor/notebooks/2774930687601032?o=7474657872577658

https://customer-academy.databricks.com/learn/courses/3962/developing-applications-with-apache-spark/lessons/73303:4207/working-with-complex-data

https://community.databricks.com/t5/learning-events/advanced-learning-festival-15-june-06-july-2026/ev-p/156932

https://community.databricks.com/t5/learning-events/virtual-learning-festival-10-october-31-october-2025/ev-p/127652

## A. DataFrame Join Operations

Joins combine two DataFrames based on matching key column(s):

* Multiple join types supported: inner, outer, left, right, cross
* The join condition can use single or multiple columns
* Column naming conflicts are handled with DataFrame aliases or by renaming columns before the join

| Join Type (`how=`) | Returns |
| :--- | :--- |
| `_inner_` (default) | Rows with matching keys in both DataFrames |
| `_left_` | All left rows, with nulls where the right side has no match |
| `_right_` | All right rows, with nulls where the left side has no match |
| `_full_outer_` | All rows from both sides, with nulls for missing matches |
| `_left_semi_` | Left rows that have a match (no right columns) |
| `_left_anti_` | Left rows that have no match |
| `_cross_` | Every combination of rows (use with care) |

In [0]:
%python
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

# 1. Criar dados de exemplo para utilizadores e encomendas
users_data = [
    (1, "Alice", "North"),
    (2, "Bob", "South"),
    (3, "Charlie", "East"),
    (4, "Diana", "West")
]
users_columns = ["user_id", "name", "region"]
users_df = spark.createDataFrame(users_data, users_columns)
display(users_df)

user_id,name,region
1,Alice,North
2,Bob,South
3,Charlie,East
4,Diana,West


In [0]:


orders_data = [
    (101, 1, "North", 250.0),
    (102, 2, "South", 150.0),
    (103, 1, "North", 300.0),
    (104, 5, "Central", 400.0) # user_id 5 não existe em users_df
]
orders_columns = ["order_id", "user_id", "region", "amount"]
orders_df = spark.createDataFrame(orders_data, orders_columns)
display(orders_df)


order_id,user_id,region,amount
101,1,North,250.0
102,2,South,150.0
103,1,North,300.0
104,5,Central,400.0


In [0]:


print("--- 1. Inner Join (Padrão) ---")
# Devolve apenas as linhas com correspondência em ambos os DataFrames
inner_join_df = users_df.join(orders_df, "user_id")
display(inner_join_df)

print("--- 2. Left Join (Mantém todos os utilizadores) ---")
# Especificando o tipo de join e usando condições explícitas
left_join_df = users_df.join(
    orders_df,
    on=users_df.user_id == orders_df.user_id,
    how="left"
)
display(left_join_df)

print("--- 3. Múltiplas Condições com Aliases ---")
# Juntando utilizando múltiplos critérios (ex: ID e Região correspondentes)
multi_cond_df = users_df.alias("u").join(
    orders_df.alias("o"),
    [
        col("u.user_id") == col("o.user_id"),
        col("u.region") == col("o.region")
    ],
    "inner"
)
display(multi_cond_df)

--- 1. Inner Join (Padrão) ---


user_id,name,region,order_id,region,amount
1,Alice,North,101,North,250.0
2,Bob,South,102,South,150.0
1,Alice,North,103,North,300.0


--- 2. Left Join (Mantém todos os utilizadores) ---


user_id,name,region,order_id,user_id,region,amount
1,Alice,North,103,1,North,300.0
1,Alice,North,101,1,North,250.0
2,Bob,South,102,2,South,150.0
3,Charlie,East,null,null,null,null
4,Diana,West,null,null,null,null


--- 3. Múltiplas Condições com Aliases ---


user_id,name,region,order_id,user_id,region,amount
1,Alice,North,101,1,North,250.0
2,Bob,South,102,2,South,150.0
1,Alice,North,103,1,North,300.0


## B. Set Operations

Set operations treat DataFrames as sets of rows, helping you understand relationships within datasets. Given $A = [1, 2, 3]$ and $B = [2, 3, 4]$:

* **UNION** - includes all unique elements from both sets, no duplicates: `A UNION B -> [1, 2, 3, 4]`
* **INTERSECTION** - includes elements in both sets: `A INTERSECT B -> [2, 3]`
* **SUBTRACTION** - includes elements in A only: `A MINUS B -> [1]`

All DataFrame set operations require matching schemas. The methods:

| Operation | Description | Example |
| :--- | :--- | :--- |
| `_union()_` | Combines all rows from both DataFrames by position | `df1.union(df2)` |
| `_unionByName()_` | Combines DataFrames matching by column names | `df1.unionByName(df2)` |
| `_intersect()_` | Returns only rows present in both DataFrames | `df1.intersect(df2)` |
| `_subtract()_` | Returns rows in the first DataFrame but not in the second | `df1.subtract(df2)` |

> **Duplicates**
> `union()` in Spark behaves like SQL `UNION ALL` - it keeps duplicates (deduplicate with `distinct()`). `intersect()` and `subtract()` remove duplicates; use the `intersectAll()` and `exceptAll()` variants to preserve them.

## C. Join Performance Considerations

Optimizing relational operations in a distributed environment:

* **Join strategy selection** - Spark automatically chooses the join strategy; reference the smaller DataFrame first where practical
* **Broadcast joins** - Spark can avoid shuffling the large side entirely by copying a small DataFrame to every Executor; use the `broadcast()` hint to request this explicitly

In [0]:
%python
from pyspark.sql import SparkSession
from pyspark.sql.functions import broadcast

# 1. Criar um DataFrame pequeno (ex: categorias de produtos)
small_data = [(1, "Electronics"), (2, "Clothing"), (3, "Home")]
small_df = spark.createDataFrame(small_data, ["category_id", "category_name"])

# 2. Criar um DataFrame maior (ex: transações de vendas)
large_data = [
    (101, 1, 150.0),
    (102, 2, 45.0),
    (103, 1, 300.0),
    (104, 3, 120.0),
    (105, 2, 85.0)
]
large_df = spark.createDataFrame(large_data, ["transaction_id", "category_id", "amount"])

display(small_df)
display(large_df)

category_id,category_name
1,Electronics
2,Clothing
3,Home


transaction_id,category_id,amount
101,1,150.0
102,2,45.0
103,1,300.0
104,3,120.0
105,2,85.0


In [0]:
%python
# Good - small_df is smaller
small_df.join(large_df, "key")

# Less efficient - large_df is larger
large_df.join(small_df, "key")

# Broadcasting a small DataFrame
from pyspark.sql.functions import broadcast
large_df.join(broadcast(small_df), "key")

In [0]:

print("--- 1. Boa prática: Referenciar o DataFrame pequeno primeiro ---")
good_join_df = small_df.join(large_df, "category_id")
display(good_join_df)

print("--- 2. Menos eficiente: Referenciar o DataFrame grande primeiro ---")
less_efficient_df = large_df.join(small_df, "category_id")
display(less_efficient_df)

print("--- 3. Broadcast Join: Enviar cópia do DataFrame pequeno para todos os executores ---")
broadcast_join_df = large_df.join(broadcast(small_df), "category_id")
display(broadcast_join_df)

--- 1. Boa prática: Referenciar o DataFrame pequeno primeiro ---


category_id,category_name,transaction_id,amount
1,Electronics,101,150.0
2,Clothing,102,45.0
1,Electronics,103,300.0
3,Home,104,120.0
2,Clothing,105,85.0


--- 2. Menos eficiente: Referenciar o DataFrame grande primeiro ---


category_id,transaction_id,amount,category_name
1,101,150.0,Electronics
2,102,45.0,Clothing
1,103,300.0,Electronics
3,104,120.0,Home
2,105,85.0,Clothing


--- 3. Broadcast Join: Enviar cópia do DataFrame pequeno para todos os executores ---


category_id,transaction_id,amount,category_name
1,101,150.0,Electronics
2,102,45.0,Clothing
1,103,300.0,Electronics
3,104,120.0,Home
2,105,85.0,Clothing
